In [ ]:
#full poster aligned code

import pandas as pd
import numpy as np
from scipy.stats import randint
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
from sklearn.feature_selection import VarianceThreshold, RFE
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

RANDOM_STATE = 42

# 1. Load Dataset (Selected Features Only)
print("Loading dataset...")

LEAKAGE_FEATURES = ["NACCMOCA", "INDEPEND", "NACCLIVS", "RESIDENC"]

selected_columns = ["SEX", "HISPANIC", "HISPOR", "RACE", "RACEX", "RACESEC", "RACESECX", "RACETER", "RACETERX",
                    "PRIMLANG", "PRIMLANX", "EDUC", "MARISTAT", "NACCLIVS", "INDEPEND", "RESIDENC", "HANDED", "NACCFAM",
                    "NACCMOM", "NACCDAD", "NACCAM", "NACCAMX", "NACCAMS", "NACCOM", "NACCFADM", "NACCFFTD", "ANYMEDS",
                    "TOBAC100", "TOBAC30", "PACKSPER", "ALCOCCAS", "ALCFREQ", "CVHATT", "HATTMULT", "CVAFIB", "CVANGIO",
                    "CVBYPASS", "CVPACDEF", "CVPACE", "CVCHF", "CVANGINA", "CVHVALVE", "CVOTHR", "CBSTROKE", "STROKMUL",
                    "PD", "SEIZURES", "NACCTBI", "TBI", "TBIBRIEF", "TRAUMBRF", "TBIEXTEN", "TRAUMEXT", "TBIWOLOS", "TRAUMCHR",
                    "DIABETES", "DIABTYPE", "HYPERTEN", "HYPERCHO", "B12DEF", "THYROID", "ARTHRIT", "ARTHTYPE", "ARTHUPEX",
                    "ARTHLOEX", "ARTHSPIN", "ARTHUNK", "INCONTU", "INCONTF", "APNEA", "RBD", "INSOMN", "OTHSLEEP", "ALCOHOL",
                    "ABUSOTHR", "ABUSX", "PTSD", "BIPOLAR", "SCHIZ", "DEP2YRS", "DEPOTHR", "ANXIETY", "OCD", "NPSYDEV",
                    "PSYCDIS", "HEIGHT", "WEIGHT", "BPSYS", "BPDIAS", "HRATE", "VISION", "VISCORR", "VISWCORR", "HEARING",
                    "HEARAID", "HEARWAID", "NACCAGE", "NACCNIHR", "NACCBMI", "NACCMOCA", "NACCNE4S", "THYDIS", "NACCUDSD"]

selected_columns = [c for c in selected_columns if c not in LEAKAGE_FEATURES]

load_columns = list(dict.fromkeys(selected_columns + ["NACCID"]))

df = pd.read_csv("/kaggle/input/naac-alzimers/investigator_nacc68 (1).csv", usecols=load_columns)

missing_values = [8888, 9999, 888, 999, 9, -4.4, -4]
df.replace(missing_values, np.nan, inplace=True)

# 2. Age Filter (poster: drop records with NACCAGE < 60)

before_age_filter = df.shape[0]
df = df[df["NACCAGE"] >= 60].copy()
print(f"Dropped {before_age_filter - df.shape[0]} rows with NACCAGE < 60 (or missing age).")

df = df.dropna(subset=["NACCUDSD"])
df["NACCUDSD_BIN"] = df["NACCUDSD"].map(lambda x: 1 if x == 4 else (0 if x in [1, 2, 3] else np.nan))
df = df.dropna(subset=["NACCUDSD_BIN"])

y = df["NACCUDSD_BIN"].astype(int)
groups = df["NACCID"]
X = df.drop(columns=["NACCUDSD", "NACCUDSD_BIN", "NACCID"])
X = X.select_dtypes(exclude=["object"])

print(f"Rows: {X.shape[0]} | Unique patients: {groups.nunique()} | Features loaded: {X.shape[1]}")

# 3. Patient-Level Train / Val / Test Split
print("Splitting dataset into train, validation, and test sets (patient-level)...")
def stratified_group_split(X, y, groups, test_fraction, random_state=RANDOM_STATE):
    n_splits = max(2, round(1 / test_fraction))
    sgkf = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    keep_idx, holdout_idx = next(sgkf.split(X, y, groups=groups))
    return keep_idx, holdout_idx


train_idx, temp_idx = stratified_group_split(X, y, groups, test_fraction=0.30)
X_train = X.iloc[train_idx].reset_index(drop=True)
y_train = y.iloc[train_idx].reset_index(drop=True)
groups_train = groups.iloc[train_idx].reset_index(drop=True)

X_temp = X.iloc[temp_idx].reset_index(drop=True)
y_temp = y.iloc[temp_idx].reset_index(drop=True)
groups_temp = groups.iloc[temp_idx].reset_index(drop=True)

val_idx, test_idx = stratified_group_split(X_temp, y_temp, groups_temp, test_fraction=0.50)
X_val, y_val = X_temp.iloc[val_idx].reset_index(drop=True), y_temp.iloc[val_idx].reset_index(drop=True)
X_test, y_test = X_temp.iloc[test_idx].reset_index(drop=True), y_temp.iloc[test_idx].reset_index(drop=True)

# Sanity check: confirm no patient appears in more than one split
train_ids = set(groups_train)
val_ids = set(groups_temp.iloc[val_idx])
test_ids = set(groups_temp.iloc[test_idx])
assert not (train_ids & val_ids), "Patient overlap between train and val!"
assert not (train_ids & test_ids), "Patient overlap between train and test!"
assert not (val_ids & test_ids), "Patient overlap between val and test!"

print(f"Train rows: {X_train.shape[0]} | Val rows: {X_val.shape[0]} | Test rows: {X_test.shape[0]}")
print(f"Train patients: {len(train_ids)} | Val patients: {len(val_ids)} | Test patients: {len(test_ids)}")
print(f"Train class distribution:\n{y_train.value_counts()}")

# 4. Drop Features with >50% Missing Values (computed on TRAIN only)
print("Dropping features with >50% missing values (train-derived threshold)...")
missing_percent = X_train.isna().mean()
keep_columns = missing_percent[missing_percent <= 0.5].index.tolist()

X_train = X_train[keep_columns]
X_val = X_val[keep_columns]
X_test = X_test[keep_columns]
print(f"Features Remaining After Dropping >50% Missing: {X_train.shape[1]}")

# 5. Handle Missing Values (Imputation, fit on train only)
print("Performing missing value imputation...")
imputer = IterativeImputer(max_iter=30, tol=1e-4, random_state=RANDOM_STATE)
X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_val = pd.DataFrame(imputer.transform(X_val), columns=X_train.columns)
X_test = pd.DataFrame(imputer.transform(X_test), columns=X_train.columns)

# 6. Apply Standardization
print("Applying standardization...")
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_val = pd.DataFrame(scaler.transform(X_val), columns=X_train.columns)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_train.columns)

# 7. Remove Low-Variance Features
print("Removing low-variance features...")
var_thresh = VarianceThreshold(threshold=0.01)
X_train_var = pd.DataFrame(var_thresh.fit_transform(X_train), columns=X_train.columns[var_thresh.get_support()])
selected_features = X_train_var.columns
X_val_var = pd.DataFrame(var_thresh.transform(X_val), columns=selected_features)
X_test_var = pd.DataFrame(var_thresh.transform(X_test), columns=selected_features)
print(f"Features After Variance Check: {len(selected_features)}")

# 8. Feature Selection: compare RF Feature Importance vs RFE
print("Computing standalone Random Forest feature importances...")
rf_importance_model = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rf_importance_model.fit(X_train_var, y_train)
importance_ranking = pd.Series(
    rf_importance_model.feature_importances_, index=X_train_var.columns
).sort_values(ascending=False)
print("Top 20 features by RF importance:")
print(importance_ranking.head(20))
print("Performing Recursive Feature Elimination (RFE)...")
rf_rfe = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
n_features_to_select = max(1, int(X_train_var.shape[1] * 0.30))
rfe = RFE(estimator=rf_rfe, n_features_to_select=n_features_to_select, step=0.10)
rfe.fit(X_train_var, y_train)

selected_features = X_train_var.columns[rfe.support_].tolist()
top_importance_features = set(importance_ranking.head(n_features_to_select).index)
overlap = top_importance_features & set(selected_features)
print(f"Overlap between top-{n_features_to_select} RF-importance features and RFE-selected features: "
      f"{len(overlap)}/{len(selected_features)}")

X_train_selected = X_train_var[selected_features]
X_val_selected = X_val_var[selected_features]
X_test_selected = X_test_var[selected_features]

print(f"Features after RFE: {len(selected_features)}")
print("Selected Features:", selected_features)

# 9. Remove Highly Correlated Features (Pearson AND Spearman)
print("Removing highly correlated features (Pearson + Spearman)...")
corr_pearson = X_train_selected.corr(method="pearson").abs()
corr_spearman = X_train_selected.corr(method="spearman").abs()

upper_pearson = corr_pearson.where(np.triu(np.ones(corr_pearson.shape), k=1).astype(bool))
upper_spearman = corr_spearman.where(np.triu(np.ones(corr_spearman.shape), k=1).astype(bool))

high_corr_pearson = [c for c in upper_pearson.columns if any(upper_pearson[c] > 0.85)]
high_corr_spearman = [c for c in upper_spearman.columns if any(upper_spearman[c] > 0.85)]
high_corr_features = sorted(set(high_corr_pearson) | set(high_corr_spearman))

X_train_final = X_train_selected.drop(columns=high_corr_features)
X_val_final = X_val_selected.drop(columns=high_corr_features, errors="ignore")
X_test_final = X_test_selected.drop(columns=high_corr_features, errors="ignore")

print(f"Final Features After Collinearity Check: {X_train_final.shape[1]}")
print("Final Selected Features:", list(X_train_final.columns))

# 10. SMOTE + Random Forest inside a Pipeline, tuned with RandomizedSearchCV using patient-grouped k-fold CV
print("Running RandomizedSearchCV with grouped k-fold CV (SMOTE applied per-fold)...")

pipeline = ImbPipeline([
    ("smote", SMOTE(random_state=RANDOM_STATE)),
    ("rf", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
])

param_distributions = {
    "rf__n_estimators": randint(100, 400),
    "rf__max_depth": [None, 5, 10, 15, 20, 30],
    "rf__min_samples_split": randint(2, 10),
    "rf__min_samples_leaf": randint(1, 5),
    "rf__max_features": ["sqrt", "log2", None],
}

cv_splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=30,         
    scoring="roc_auc",
    cv=cv_splitter,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    refit=True,
    verbose=1,
)

search.fit(X_train_final, y_train, groups=groups_train)

print("Best hyperparameters:", search.best_params_)
print("Best CV ROC-AUC (grouped k-fold, held out per fold):", search.best_score_)

best_model = search.best_estimator_ 

# 11. Evaluate on Validation & Test
print("\n--- Validation Performance ---")
y_val_pred = best_model.predict(X_val_final)
y_val_proba = best_model.predict_proba(X_val_final)[:, 1]
print(classification_report(y_val, y_val_pred))
print(confusion_matrix(y_val, y_val_pred))
print(f"Validation ROC-AUC: {roc_auc_score(y_val, y_val_proba):.4f}")

print("\n--- Test Performance ---")
y_test_pred = best_model.predict(X_test_final)
y_test_proba = best_model.predict_proba(X_test_final)[:, 1]
print(classification_report(y_test, y_test_pred))
print(confusion_matrix(y_test, y_test_pred))
print(f"Test ROC-AUC: {roc_auc_score(y_test, y_test_proba):.4f}")